## Naive Bayes from Scratch: Machine Fault Detection

## Dataset

Each row contains sensor indicators collected during one machine inspection. The target class is `normal` or `fault`. All input features are binary: `1` means the condition is present and `0` means it is absent.

| Feature | Meaning |
|---|---|
| `high_temperature` | The operating temperature is above the safe threshold. |
| `strong_vibration` | The vibration level is unusually high. |
| `low_pressure` | The system pressure is below the expected range. |
| `low_oil_level` | The lubricant level is low. |
| `warning_light_on` | The machine warning light is on. |
| `unusual_noise` | The machine produces an unusual noise. |

The `split` column provides a fixed train/test split.

In [22]:
%pip install numpy pandas


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [23]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

columns = [
    "high_temperature",
    "strong_vibration",
    "low_pressure",
    "low_oil_level",
    "warning_light_on",
    "unusual_noise",
    "label",
    "split",
]

rows = [
    [1, 1, 1, 0, 1, 1, "fault", "train"],
    [1, 1, 0, 1, 1, 1, "fault", "train"],
    [1, 0, 1, 1, 1, 0, "fault", "train"],
    [0, 1, 1, 0, 1, 1, "fault", "train"],
    [1, 1, 0, 0, 1, 0, "fault", "train"],
    [0, 1, 0, 1, 1, 1, "fault", "train"],
    [1, 0, 1, 0, 0, 1, "fault", "train"],
    [1, 1, 1, 1, 1, 1, "fault", "train"],
    [0, 0, 1, 1, 1, 0, "fault", "train"],
    [1, 1, 0, 1, 0, 1, "fault", "train"],
    [0, 1, 1, 0, 0, 1, "fault", "train"],
    [1, 0, 0, 1, 1, 0, "fault", "train"],
    [0, 0, 0, 0, 0, 0, "normal", "train"],
    [0, 0, 0, 0, 1, 0, "normal", "train"],
    [0, 1, 0, 0, 0, 0, "normal", "train"],
    [1, 0, 0, 0, 0, 0, "normal", "train"],
    [0, 0, 1, 0, 0, 0, "normal", "train"],
    [0, 0, 0, 1, 0, 0, "normal", "train"],
    [0, 1, 0, 0, 0, 1, "normal", "train"],
    [1, 0, 0, 0, 1, 0, "normal", "train"],
    [0, 0, 0, 0, 0, 1, "normal", "train"],
    [0, 0, 1, 0, 1, 0, "normal", "train"],
    [0, 1, 0, 1, 0, 0, "normal", "train"],
    [1, 0, 0, 1, 0, 0, "normal", "train"],
    [1, 1, 1, 1, 1, 1, "fault", "test"],
    [1, 0, 1, 1, 1, 0, "fault", "test"],
    [0, 1, 1, 0, 1, 1, "fault", "test"],
    [1, 1, 0, 0, 0, 1, "fault", "test"],
    [0, 0, 0, 0, 0, 0, "normal", "test"],
    [0, 1, 0, 0, 0, 0, "normal", "test"],
    [1, 0, 0, 0, 1, 0, "normal", "test"],
    [0, 0, 1, 1, 0, 0, "normal", "test"],
]

data = pd.DataFrame(rows, columns=columns)
data

,high_temperature,strong_vibration,low_pressure,low_oil_level,warning_light_on,unusual_noise,label,split
0,1,1,1,0,1,1,fault,train
1,1,1,0,1,1,1,fault,train
2,1,0,1,1,1,0,fault,train
3,0,1,1,0,1,1,fault,train
4,1,1,0,0,1,0,fault,train
5,0,1,0,1,1,1,fault,train
6,1,0,1,0,0,1,fault,train
7,1,1,1,1,1,1,fault,train
8,0,0,1,1,1,0,fault,train
9,1,1,0,1,0,1,fault,train


In [24]:
feature_columns = [
    "high_temperature",
    "strong_vibration",
    "low_pressure",
    "low_oil_level",
    "warning_light_on",
    "unusual_noise",
]

train_data = data[data["split"] == "train"].reset_index(drop=True)
test_data = data[data["split"] == "test"].reset_index(drop=True)

X_train = train_data[feature_columns]
y_train = train_data["label"]
X_test = test_data[feature_columns]
y_test = test_data["label"]

print("Training shape:", X_train.shape)
print("Test shape:    ", X_test.shape)
print("\nTraining class counts:")
print(y_train.value_counts())

Training shape: (24, 6)
Test shape:     (8, 6)

Training class counts:
label
fault     12
normal    12
Name: count, dtype: int64


## Calculate Class Counts and Prior Probabilities

For each class $c$, calculate the prior probability:

$$P(c) = \frac{N_c}{N}$$

Store the class counts in a Pandas `Series` named `class_counts` and the prior probabilities in a Pandas `Series` named `priors`. The index of both Series must contain all class labels.

In [25]:
classes = np.sort(y_train.unique())

class_counts = y_train.value_counts().sort_index()
priors = class_counts / len(y_train)

priors

label
fault     0.5
normal    0.5
Name: count, dtype: float64

## Likelihoods with Laplace smoothing 

Treat each input column as a discrete feature. For feature $X_j$, observed value $v$, and class $c$, estimate:

$$P(X_j=v \mid C=c) = \frac{N_{j,v,c}+\alpha}{N_c+\alpha K_j}$$

where $N_{j,v,c}$ is the number of matching training examples, $N_c$ is the number of training examples in class $c$, and $K_j$ is the number of possible values of feature $X_j$ observed in the training set.

Complete `estimate_likelihoods`. The returned DataFrame must use classes as rows and `(feature, value)` pairs as MultiIndex columns. Use $\alpha=1$.

In [26]:
def estimate_likelihoods(X, y, class_labels, alpha=1.0):
    """Estimate feature likelihoods for every class.

    Parameters
    ----------
    X : pandas.DataFrame
        Discrete training features.
    y : pandas.Series
        Training labels aligned with X.
    class_labels : array-like
        Ordered class labels to estimate.
    alpha : float
        Laplace smoothing parameter.

    Returns
    -------
    pandas.DataFrame
        Rows are classes and MultiIndex columns are (feature, value).
    """
    possible_values = {
        feature: np.sort(X[feature].unique()) for feature in X.columns
    }
    likelihood_columns = pd.MultiIndex.from_tuples(
        [
            (feature, value)
            for feature, values in possible_values.items()
            for value in values
        ],
        names=["feature", "value"],
    )
    likelihoods = pd.DataFrame(
        index=class_labels, columns=likelihood_columns, dtype=float
    )

    for class_label in class_labels:
        X_class = X[y == class_label]

        for feature, values in possible_values.items():
            number_of_values = len(values)

            for value in values:
                value_count = (X_class[feature] == value).sum()
                likelihood = (value_count + alpha) / (len(X_class) + alpha * number_of_values)  
                likelihoods.loc[class_label, (feature, value)] = likelihood

    return likelihoods


alpha = 1.0
likelihoods = estimate_likelihoods(
    X_train, y_train, classes, alpha=alpha
)
likelihoods

feature high_temperature           strong_vibration           low_pressure  \
value                  0         1                0         1            0   
fault           0.357143  0.642857         0.357143  0.642857     0.428571   
normal          0.714286  0.285714         0.714286  0.285714     0.785714   

feature           low_oil_level           warning_light_on            \
value           1             0         1                0         1   
fault    0.571429      0.428571  0.571429         0.285714  0.714286   
normal   0.214286      0.714286  0.285714         0.714286  0.285714   

feature unusual_noise            
value               0         1  
fault        0.357143  0.642857  
normal       0.785714  0.214286

## Log-scores and predictions

For every test example and candidate class, calculate the unnormalized log-posterior score:

$$\log score(c) = \log P(C=c) + \sum_j \log P(X_j=x_j \mid C=c)$$

Complete `calculate_log_scores`, then predict the class with the largest log-score for each test example. Store the scores in `test_log_scores` and the predictions in a one-dimensional NumPy array named `y_pred`.

In [27]:
def calculate_log_scores(X, class_labels, class_priors, feature_likelihoods):
    """Calculate one Naive Bayes log-score per class and example.

    Parameters
    ----------
    X : pandas.DataFrame
        Examples to score.
    class_labels : array-like
        Ordered class labels.
    class_priors : pandas.Series
        Prior probability of each class.
    feature_likelihoods : pandas.DataFrame
        Conditional probabilities with MultiIndex columns.

    Returns
    -------
    pandas.DataFrame
        Unnormalized log-posterior scores for all rows and classes.
    """
    scores = pd.DataFrame(index=X.index, columns=class_labels, dtype=float)

    for row_index, row in X.iterrows():
        for class_label in class_labels:
            log_score = np.log(class_priors[class_label])

            for feature in X.columns:
                observed_value = row[feature]

                probability = feature_likelihoods.loc[class_label, 
                                                      (feature, observed_value)]
                log_score += np.log(probability)

            scores.loc[row_index, class_label] = log_score

    return scores


test_log_scores = calculate_log_scores(
    X_test, classes, priors, likelihoods
)

y_pred = test_log_scores.idxmax(axis=1)

prediction_results = X_test.copy()
prediction_results["actual"] = y_test.to_numpy()
prediction_results["predicted"] = y_pred
prediction_results

,high_temperature,strong_vibration,low_pressure,low_oil_level,warning_light_on,unusual_noise,actual,predicted
0,1,1,1,1,1,1,fault,fault
1,1,0,1,1,1,0,fault,fault
2,0,1,1,0,1,1,fault,fault
3,1,1,0,0,0,1,fault,fault
4,0,0,0,0,0,0,normal,normal
5,0,1,0,0,0,0,normal,normal
6,1,0,0,0,1,0,normal,normal
7,0,0,1,1,0,0,normal,normal


## Model evaluation

1. Calculate accuracy from scratch.
2. Complete `confusion_matrix_from_scratch`. Rows must represent **actual** classes and columns must represent **predicted** classes.
3. Do not use a metrics library.

$$Accuracy = \frac{\text{number of correct predictions}}{\text{number of test examples}}$$

In [28]:
def confusion_matrix_from_scratch(y_true, y_predicted, class_labels):
    """Build a confusion matrix without using a metrics library.

    Parameters
    ----------
    y_true : array-like
        Actual class labels.
    y_predicted : array-like
        Predicted class labels.
    class_labels : array-like
        Row and column order for the matrix.

    Returns
    -------
    pandas.DataFrame
        Confusion matrix with actual rows and predicted columns.
    """
    matrix = np.zeros((len(class_labels), len(class_labels)), dtype=int)
    label_to_position = {
        label: position for position, label in enumerate(class_labels)
    }

    for actual, predicted in zip(y_true, y_predicted):
        actual_position = label_to_position[actual]
        predicted_position = label_to_position[predicted]
        matrix[actual_position, predicted_position] += 1

    return pd.DataFrame(
        matrix,
        index=[f"actual_{label}" for label in class_labels],
        columns=[f"predicted_{label}" for label in class_labels],
    )


# TODO
accuracy = np.mean(y_test == y_pred)
confusion = confusion_matrix_from_scratch(y_test, y_pred, classes)

print(f"Accuracy: {accuracy:.3f}")
display(confusion)

Accuracy: 1.000


,predicted_fault,predicted_normal
actual_fault,4,0
actual_normal,0,4
